# Installing important libraries

In [1]:
%pip install openai langchain faiss-cpu pypdf tiktoken docarray PyPDF tiktoken langchain-openai flashrank langchain-community pillow sentence-transformers langchain-docling accelerate
# langchain-docling

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os
import time
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
# from langchain_docling import DoclingLoader


c:\Users\rocky\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

True

In [4]:
import glob
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

pdf_files = glob.glob("Policy+Documents/*.pdf")

loader = DoclingLoader(
    file_path=pdf_files,
    export_type=ExportType.DOC_CHUNKS
)

documents = loader.load()

2025-11-18 00:14:56,184 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-11-18 00:14:56,222 - INFO - Going to convert document batch...
2025-11-18 00:14:56,223 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2025-11-18 00:14:56,246 - INFO - Loading plugin 'docling_defaults'
2025-11-18 00:14:56,248 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2025-11-18 00:14:56,249 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-11-18 00:14:56,265 - INFO - Loading plugin 'docling_defaults'
2025-11-18 00:14:56,269 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2025-11-18 00:14:56,270 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-11-18 00:14:56,547 - INFO - Accelerator device: 'cpu'
[INFO] 2025-11-18 0

In [5]:

# Load PDF documents with error handling
# print("Loading PDF documents from ./Policy+Documents...")
# try:
#     pdf_directory_loader = PyPDFDirectoryLoader("./Policy+Documents")
#     documents = pdf_directory_loader.load()
#     print(f"✓ Successfully loaded {len(documents)} documents")
#     print(f"✓ Total pages: {sum(doc.metadata.get('total_pages', 1) for doc in documents)}")
# except Exception as e:
#     print(f"Error loading documents: {str(e)}")
#     raise

In [6]:
documents[0].page_content[:100]

"- <<Date>>\n- <<Policyholder's Name>>\n- <<Policyholder's Address>>\n- <<Policyholder's Contact Number>"

In [7]:
# Split documents into chunks
print("Splitting documents into chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
splits = text_splitter.split_documents(documents)
print(f"✓ Created {len(splits)} document chunks")
print(f"✓ Average chunk size: {sum(len(s.page_content) for s in splits) // len(splits)} characters")

Splitting documents into chunks...
✓ Created 1136 document chunks
✓ Average chunk size: 565 characters


In [8]:
print(splits[0])

page_content='- <<Date>>
- <<Policyholder's Name>>
- <<Policyholder's Address>>
- <<Policyholder's Contact Number>>
Dear <<Policyholder's Name>>,' metadata={'source': 'Policy+Documents\\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/0', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 1, 'bbox': {'l': 72.0, 't': 766.5540649804688, 'r': 115.926, 'b': 758.3620649804687, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 8]}]}, {'self_ref': '#/texts/1', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 1, 'bbox': {'l': 72.0, 't': 755.0340649804688, 'r': 182.527, 'b': 746.8420649804688, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 23]}]}, {'self_ref': '#/texts/2', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'cont

In [9]:
# Initialize embeddings model with timeout and retry settings
print("Initializing OpenAI embeddings model...")
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",  # Using smaller, faster model
    request_timeout=60,  # 60 second timeout
    max_retries=3  # Retry up to 3 times on failure
)
print("✓ Embeddings model initialized")

Initializing OpenAI embeddings model...
✓ Embeddings model initialized


In [10]:
# Test embedding on a single document
print("Testing embeddings on a sample chunk...")
try:
    test_embedding = embeddings_model.embed_documents([splits[0].page_content])
    print(f"✓ Test embedding successful - dimension: {len(test_embedding[0])}")
except Exception as e:
    print(f"✗ Error testing embeddings: {str(e)}")
    raise

Testing embeddings on a sample chunk...


2025-11-18 00:33:42,290 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


✓ Test embedding successful - dimension: 1536


In [11]:
from langchain_classic.embeddings import CacheBackedEmbeddings  
from langchain_classic.storage import LocalFileStore 
store = LocalFileStore("./cache/") 

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embeddings_model,
    store,
    namespace="semantic-spotter"
)

c:\Users\rocky\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [12]:
# Preview first few splits (safe check)
print("Preview of document splits:")
try:
    for i, split in enumerate(splits[:3]):
        print(f"\n--- Split {i+1} ---")
        print(f"Source: {split.metadata.get('source', 'Unknown')}")
        print(f"Page: {split.metadata.get('page', 'Unknown')}")
        print(f"Content preview: {split.page_content[:150]}...")
    print(f"\n✓ Total splits available: {len(splits)}")
except Exception as e:
    print(f"Error previewing splits: {e}")
    raise

Preview of document splits:

--- Split 1 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: Unknown
Content preview: - <<Date>>
- <<Policyholder's Name>>
- <<Policyholder's Address>>
- <<Policyholder's Contact Number>>
Dear <<Policyholder's Name>>,...

--- Split 2 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: Unknown
Content preview: Sub: Your Policy no. <<  >>
We are glad to inform you that your proposal has been accepted and the HDFC Life Easy Health ('Policy') being this documen...

--- Split 3 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: Unknown
Content preview: Policy document:
As an evidence of the insurance contract between HDFC Life Insurance Company Limited and you, the Policy is  enclosed herewith. Pleas...

✓ Total splits available: 1136


In [13]:

def create_vector_store_faiss(splits, embeddings_model, save_path="./faiss_store"):
    """Create and save FAISS vector store."""
    print(f"Creating vector store from {len(splits)} documents...")
    start_time = time.time()
    
    try:
        # Create FAISS store directly from documents
        if os.path.exists(save_path):
            return FAISS.load_local(save_path, embeddings=embeddings_model, allow_dangerous_deserialization=True)
        
        vectordb = FAISS.from_documents(
            documents=splits,
            embedding=embeddings_model
        )
        print(f"✓ FAISS vector store created")
        
        # Save to disk
        os.makedirs(save_path, exist_ok=True)
        vectordb.save_local(save_path)
        print(f"✓ Saved to: {save_path}")
        
        elapsed = time.time() - start_time
        print(f"✓ Time: {elapsed:.1f}s ({elapsed/60:.1f}m)")
        
        return vectordb
    except Exception as e:
        print(f"✗ Error: {type(e).__name__}: {e}")
        raise


In [14]:
# Create the vector store
try:
    vectordb = create_vector_store_faiss(splits, cached_embedder, "./faiss_store")
    print("✓ Vector store ready for similarity search")
except Exception as e:
    print(f"Failed to create vector store: {str(e)}")
    raise


Creating vector store from 1136 documents...


2025-11-18 00:33:48,505 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-18 00:33:52,462 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-18 00:34:01,840 - INFO - Loading faiss with AVX2 support.
2025-11-18 00:34:01,876 - INFO - Successfully loaded faiss with AVX2 support.


✓ FAISS vector store created
✓ Saved to: ./faiss_store
✓ Time: 19.6s (0.3m)
✓ Vector store ready for similarity search


In [15]:
from langchain.tools import tool
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

compressor = FlashrankRerank()
compression_retriever = None

if 'vectordb' in globals() and vectordb is not None:
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=vectordb.as_retriever(search_kwargs={"k": 20})
    )

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query"""


    if 'vectordb' not in globals() and vectordb is None:
        return "No vector store available", []
    
    #prefer compression_retriever when available , otherwise use fallback to basic retriever
    try:
        if compression_retriever is not None:
            retrieved_docs = compression_retriever.invoke(
            query
        )
            
        else:
            retrieved_docs = vectordb.as_retriever(search_kwargs={"k": 5}).get_relevant_documents(query)
    # retrieved_docs = vectordb.similarity_search(query, k=2)
    # serialized = "\n\n".join(
    #     (f"Source: {doc.metadata}\n Page Content: {doc.page_content}")
    #     for doc in retrieved_docs
    # )
       
    except Exception as e:
            print("error in retrieve_context",e)

    serialized = "\n\n".join(
            f"Source: {d.metadata.get('source','unknown')} | Page: {d.metadata.get('page','?')}\n{d.page_content}"
            for d in retrieved_docs
        )
    return serialized, retrieved_docs

In [16]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

# Instantiate the LLM
llm = ChatOpenAI(model_name="gpt-4o-mini", streaming=True)

tools = [retrieve_context]

prompt = """You are an insurance policy QA assistant. Answer questions ONLY based on the policy documents provided in details.

CRITICAL OUTPUT FORMAT RULES:
1. ALWAYS respond with EXACTLY this format, nothing else:
answer: [your answer here]
source: [metadata info from retrieved document pdf name and page no.]
2. Do NOT include any explanation, preamble, or extra text
3. Do NOT repeat the question
4. Do NOT include metadata (producer, creator, page, author, etc.)
5. Use only the actual policy content
6. If answer not found, write: "Not found in the provided policy context."

EXAMPLE OUTPUT:
answer: The policy covers up to $500,000 for life insurance
"""

prompt = """You are an insurance policy QA assistant. Answer questions ONLY based on the policy documents provided in details.

CRITICAL OUTPUT FORMAT RULES:
1. ALWAYS respond with EXACTLY this format, nothing else:
answer: [your answer here]
source: [metadata info from retrieved document pdf name and page no.]

EXAMPLE OUTPUT:
answer: The policy covers up to $500,000 for life insurance
"""

agent = create_agent(llm, tools, system_prompt=prompt)


In [17]:
from langchain.messages import HumanMessage

def insurance_agent(query: str):
    response = agent.invoke({
        'messages': [
            HumanMessage(content=(
            query
            ))
        ]
    })

    print(response['messages'][-1].content)

In [18]:
insurance_agent( "What is the life insurance policy coverage amount?")

2025-11-18 00:34:05,443 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-18 00:34:06,971 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-18 00:34:10,909 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: The coverage amount for the life insurance policy, specifically defined as the Sum Assured, varies from one Scheme Member to another and is detailed in the Certificate of Insurance issued to each member. In the case of Accidental Death, the maximum benefit payable under all policies is limited to Rs.10,000,000 (Rupees 1 crore only).  
source: Policy+Documents\HDFC-Life-Group-Poorna-Suraksha-101N137V02-Policy-Document.pdf | Page: ? and Policy+Documents\HDFC-Life-Group-Term-Life-Policy.pdf | Page: ?


In [19]:
insurance_agent( "Can a 100 year plus person do a term insurance?")

2025-11-18 00:34:16,817 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-18 00:34:17,490 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-18 00:34:20,872 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: The policy documents do not specify an age limit for purchasing term insurance, thus a person over 100 years of age may still be eligible to apply depending on specific terms set by the insurer. 
source: Policy+Documents\HDFC-Life-Group-Term-Life-Policy.pdf | Page: ?


In [20]:
insurance_agent("what is the Definitions of Critical Illnesses? based on policy?")

2025-11-18 00:34:23,190 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-18 00:34:24,294 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-18 00:34:29,763 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: The definitions of critical illnesses covered under the policy include myocardial infarction, which is defined as the first occurrence of a heart attack resulting in the death of a portion of the heart muscle due to inadequate blood supply, confirmed by specific clinical criteria such as symptoms, electrocardiogram changes, and elevation of specific enzymes. Other critical illnesses include blindness, major head trauma, third-degree burns, Parkinson's disease, permanent paralysis of limbs, multiple sclerosis with persisting symptoms, motor neuron disease with permanent symptoms, benign brain tumor, and major organ transplant as a recipient.
source: Policy+Documents\HDFC-Life-Group-Poorna-Suraksha-101N137V02-Policy-Document.pdf | Page: ?


In [21]:
insurance_agent("what is the life insurance coverage for disability?")

2025-11-18 00:34:33,948 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-18 00:34:34,635 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-18 00:34:38,154 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: The policy highlights benefits for death and critical illness but does not explicitly mention life insurance coverage for disability. 
source: Policy+Documents\HDFC-Life-Group-Poorna-Suraksha-101N137V02-Policy-Document.pdf | Page: ?


In [22]:
# retrieve_context("what is the Definitions of Critical Illnesses? based on policy?")

In [23]:
# retrieve_context("Can a 100 year plus person do a term insurance?")